In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))   

from shared import config
from shared.data import get_datasets
from shared.augment import get_augmentation
from shared.evaluate import evaluate

config.set_seed()
train_ds, val_ds = get_datasets()

"""Example / template model -- ResNet50 transfer learning.

Copy this file to your own ``models/<yourname>.py`` and swap the backbone +
``preprocess_input`` for the architecture you are assigned. This file is NOT
frozen; each member owns their own model module. The ONLY hard requirement is
the function signature below, which train.py depends on::

    build_model(num_classes: int, augmentation: keras.Sequential) -> keras.Model

Your model MUST internally stack, in this order:
    1. the shared `augmentation` (passed in -- do not build your own),
    2. your backbone's `preprocess_input` (preprocessing travels with the model;
       the shared data loader returns RAW [0, 255] pixels on purpose),
    3. the pretrained backbone (frozen for feature extraction),
    4. a new classification head ending in softmax over `num_classes`.
"""

from __future__ import annotations

from tensorflow import keras
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from shared import config


def build_model(
    num_classes: int, augmentation: keras.Sequential
) -> keras.Model:
    """Build and compile a ResNet50 transfer-learning model.

    Args:
        num_classes: Number of output classes (``config.NUM_CLASSES``).
        augmentation: The shared training-time augmentation pipeline from
            ``shared.augment.get_augmentation`` (active only during training).

    Returns:
        A compiled ``keras.Model`` outputting softmax probabilities.
    """
    inputs = keras.Input(
        shape=(config.IMAGE_SIZE, config.IMAGE_SIZE, 3), name="image"
    )

    # 1. Shared augmentation (raw [0, 255] in, raw [0, 255] out).
    x = augmentation(inputs)

    # 2. Backbone-specific preprocessing -- different for every backbone, which
    #    is exactly why it lives here and NOT in shared/data.py.
    x = preprocess_input(x)

    # 3. Pretrained backbone as a frozen feature extractor.
    backbone = ResNet50(
        include_top=False,
        weights="imagenet",
        input_shape=(config.IMAGE_SIZE, config.IMAGE_SIZE, 3),
        pooling="avg",
    )
    backbone.trainable = False
    x = backbone(x, training=False)

    # 4. New classification head.
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="example_resnet50")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model



Found 2800 files belonging to 4 classes.
Found 400 files belonging to 4 classes.


In [ ]:
# Cell 3 — build the model, then train it
augmentation = get_augmentation()
model = build_model(config.NUM_CLASSES, augmentation)
model.summary()


Model: "example_resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image (InputLayer)  │ (None, 224, 224,  │          0 │ -                 │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ aerial_augmentation │ (None, 224, 224,  │          0 │ image[0][0]       │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_3          │ (None, 224, 224)  │          0 │ aerial_augmentat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_4          │ (None, 224, 224)  │          0 │ aerial_augmentat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_5          │ (None, 224, 224)  │          0 │ aerial_augmentat… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack_1 (Stack)     │ (None, 224, 224,  │          0 │ get_item_3[0][0], │
│                     │ 3)                │            │ get_item_4[0][0], │
│                     │                   │            │ get_item_5[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 224, 224,  │          0 │ stack_1[0][0]     │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resnet50            │ (None, 2048)      │ 23,587,712 │ add_1[0][0]       │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 2048)      │          0 │ resnet50[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 4)         │      8,196 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 8,196 (32.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)